# Worksheet 8
**Author:** Ujwal Shrestha
**Student ID:** 2461787


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error


## 1. Custom Decision Tree vs Scikit-Learn


In [2]:
class CustomDecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        if len(unique_classes) == 1:
            return {'class': unique_classes[0]}
        if num_samples == 0 or (self.max_depth and depth >= self.max_depth):
            return {'class': np.bincount(y).argmax()}

        best_info_gain = -float('inf')
        best_split = None

        for feature_idx in range(num_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                left_y = y[left_mask]
                right_y = y[right_mask]

                info_gain = self._information_gain(y, left_y, right_y)

                if info_gain > best_info_gain:
                    best_info_gain = info_gain
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'left_y': left_mask,
                        'right_y': right_mask
                    }

        if best_split is None:
             return {'class': np.bincount(y).argmax()}

        left_tree = self._build_tree(X[best_split['left_y']], y[best_split['left_y']], depth + 1)
        right_tree = self._build_tree(X[best_split['right_y']], y[best_split['right_y']], depth + 1)

        return {
            'feature_idx': best_split['feature_idx'],
            'threshold': best_split['threshold'],
            'left_tree': left_tree,
            'right_tree': right_tree
        }

    def _information_gain(self, parent, left, right):
        parent_entropy = self._entropy(parent)
        left_entropy = self._entropy(left)
        right_entropy = self._entropy(right)
        weighted_avg_entropy = (len(left) / len(parent)) * left_entropy + (len(right) / len(parent)) * right_entropy
        return parent_entropy - weighted_avg_entropy

    def _entropy(self, y):
        class_probs = np.bincount(y) / len(y)
        return -np.sum(class_probs * np.log2(class_probs + 1e-9))

    def predict(self, X):
        return [self._predict_single(x, self.tree) for x in X]

    def _predict_single(self, x, tree):
        if 'class' in tree:
            return tree['class']

        feature_val = x[tree['feature_idx']]
        if feature_val <= tree['threshold']:
            return self._predict_single(x, tree['left_tree'])
        else:
            return self._predict_single(x, tree['right_tree'])


In [3]:
data = load_iris()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

custom_tree = CustomDecisionTree(max_depth=3)
custom_tree.fit(X_train, y_train)
y_pred_custom = custom_tree.predict(X_test)
accuracy_custom = accuracy_score(y_test, y_pred_custom)
print(f"Custom Decision Tree Accuracy: {accuracy_custom:.4f}")

sklearn_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_tree.fit(X_train, y_train)
y_pred_sklearn = sklearn_tree.predict(X_test)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)
print(f"Scikit-learn Decision Tree Accuracy: {accuracy_sklearn:.4f}")

print("-" * 30)
print(f"Accuracy Comparison:")
print(f"Custom Decision Tree: {accuracy_custom:.4f}")
print(f"Scikit-learn Decision Tree: {accuracy_sklearn:.4f}")


Custom Decision Tree Accuracy: 1.0000
Scikit-learn Decision Tree Accuracy: 1.0000
------------------------------
Accuracy Comparison:
Custom Decision Tree: 1.0000
Scikit-learn Decision Tree: 1.0000


## 2. Ensemble Methods (Classification)


In [4]:
wine_data = load_wine()
X_wine = wine_data.data
y_wine = wine_data.target

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_wine, y_wine, test_size=0.2, random_state=42)

dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train_w, y_train_w)
y_pred_dt = dt_clf.predict(X_test_w)
f1_dt = f1_score(y_test_w, y_pred_dt, average='weighted')

rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train_w, y_train_w)
y_pred_rf = rf_clf.predict(X_test_w)
f1_rf = f1_score(y_test_w, y_pred_rf, average='weighted')

print(f"Decision Tree F1 Score: {f1_dt:.4f}")
print(f"Random Forest F1 Score: {f1_rf:.4f}")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train_w, y_train_w)

print(f"Best Parameters (GridSearch): {grid_search.best_params_}")
print(f"Best Cross-Val F1 Score: {grid_search.best_score_:.4f}")


Decision Tree F1 Score: 0.9440
Random Forest F1 Score: 1.0000
Best Parameters (GridSearch): {'criterion': 'gini', 'max_depth': None, 'n_estimators': 100}
Best Cross-Val F1 Score: 0.9783


## 3. Regression Model


In [ ]:
cal_housing = fetch_california_housing()
X_reg = cal_housing.data
y_reg = cal_housing.target

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train_r, y_train_r)
y_pred_dt_r = dt_reg.predict(X_test_r)
rmse_dt = np.sqrt(mean_squared_error(y_test_r, y_pred_dt_r))
print(f"Decision Tree Regressor RMSE: {rmse_dt:.4f}")

rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train_r, y_train_r)
y_pred_rf_r = rf_reg.predict(X_test_r)
rmse_rf = np.sqrt(mean_squared_error(y_test_r, y_pred_rf_r))
print(f"Random Forest Regressor RMSE: {rmse_rf:.4f}")

param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

random_search = RandomizedSearchCV(estimator=RandomForestRegressor(random_state=42), param_distributions=param_dist, n_iter=10, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, random_state=42)
random_search.fit(X_train_r, y_train_r)

print(f"Best Parameters (RandomSearch): {random_search.best_params_}")
print(f"Best Cross-Val Negative MSE: {random_search.best_score_:.4f}")


Decision Tree Regressor RMSE: 0.7037
Random Forest Regressor RMSE: 0.5053
